# 07 - Tokenization disruption and failure association

This notebook measures how Unicode perturbations alter o200k BPE and DistilBERT WordPiece sequences, then tests whether greater disruption is associated with classification failure on examples the model originally classified correctly.

**Inputs:** The attacked corpus and raw BPE-TF-IDF and DistilBERT predictions.  
**Outputs:** Tokenization measurements, clustered binomial regressions, and disruption-quartile summaries.

## Setup and disruption measures

For each clean/attacked token sequence, the notebook records token-count change, normalized Levenshtein edit distance, and multiset Jaccard overlap. Token encodings are cached by text within each tokenizer to avoid repeated work across attack conditions.

In [1]:
import math
from collections import Counter
from functools import cache
from pathlib import Path
import pandas as pd
import yaml
from rapidfuzz.distance import Levenshtein

# Same path setup as notebook 00; repeated for independent execution.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CONFIG = yaml.safe_load((ROOT / 'config/experiment.yaml').read_text(encoding='utf-8'))

PATHS = {key: ROOT / value for key, value in CONFIG['paths'].items()}
for directory in (PATHS['tables'],):
    directory.mkdir(parents=True, exist_ok=True)

def token_multiset_jaccard(left, right):
    a, b = Counter(left), Counter(right)
    denominator = (a | b).total()
    return (a & b).total() / denominator if denominator else 1.0

def tokenization_metrics(clean_tokens, attacked_tokens):
    clean_n, attacked_n = len(clean_tokens), len(attacked_tokens)
    return {
        'clean_token_count': clean_n,
        'attacked_token_count': attacked_n,
        'relative_token_count_change': (attacked_n - clean_n) / max(clean_n, 1),
        'normalized_token_edit': float(Levenshtein.normalized_distance(clean_tokens, attacked_tokens)),
        'token_multiset_jaccard': token_multiset_jaccard(clean_tokens, attacked_tokens),
    }


## Measure tokenization disruption

Both tokenizers process every attacked row with special tokens excluded from the WordPiece comparison. `relative_token_count_change` captures fragmentation or compression, `normalized_token_edit` captures sequence replacement, and `token_multiset_jaccard` captures retained token content regardless of order.

In [2]:
import tiktoken
from transformers import AutoTokenizer

attacks = pd.read_parquet(PATHS['attacks'])
bpe = tiktoken.get_encoding('o200k_base')
wordpiece = AutoTokenizer.from_pretrained(CONFIG['transformer']['checkpoint'])
tokenizers = {
    'o200k_base': lambda text: bpe.encode(text),
    'distilbert_wordpiece': lambda text: wordpiece.encode(text, add_special_tokens=False),
}

records = []
for tokenizer_name, encode in tokenizers.items():
    @cache
    def cached_encode(text):
        return encode(text)
    for row in attacks.itertuples(index=False):
        metrics = tokenization_metrics(cached_encode(row.clean_text), cached_encode(row.attacked_text))
        records.append({
            'dataset': row.dataset, 'example_id': row.example_id, 'attack': row.attack,
            'strength': float(row.strength), 'attack_seed': int(row.attack_seed),
            'changed_count': int(row.changed_count), 'tokenizer': tokenizer_name, **metrics,
        })

tokenization = pd.DataFrame.from_records(records)
tokenization.to_parquet(PATHS['tables'] / 'tokenization.parquet', index=False)
tokenization.groupby(['tokenizer', 'attack'])[['relative_token_count_change', 'normalized_token_edit', 'token_multiset_jaccard']].mean()


relative_token_count_change  \
tokenizer            attack                                       
distilbert_wordpiece compatibility                    -0.024092   
                     composition                       0.000000   
                     homoglyph                         0.438765   
                     invisible                         0.000000   
                     mixed                             0.066039   
o200k_base           compatibility                     1.459882   
                     composition                       0.000000   
                     homoglyph                         0.579771   
                     invisible                         1.205540   
                     mixed                             1.227338   

                                    normalized_token_edit  \
tokenizer            attack                                 
distilbert_wordpiece compatibility               0.440390   
                     composition                 0.000000   
                     homoglyph                   0.465307   
                     invisible                   0.000000   
                     mixed                       0.207159   
o200k_base           compatibility               0.695603   
                     composition                 0.000000   
                     homoglyph                   0.504087   
                     invisible                   0.617262   
                     mixed                       0.640037   

                                    token_multiset_jaccard  
tokenizer            attack                                 
distilbert_wordpiece compatibility                0.430456  
                     composition                  1.000000  
                     homoglyph                    0.463459  
                     invisible                    1.000000  
                     mixed                        0.717761  
o200k_base           compatibility                0.265672  
                     composition                  1.000000  
                     homoglyph                    0.433160  
                     invisible                    0.343567  
                     mixed                        0.318761

## Associate disruption with model failure

Analysis is restricted to effective attacks on examples that were correct when clean. For each model, a binomial regression predicts failure using standardized token edit and count change, controlling for dataset, attack family, strength, and Transformer seed where applicable. Standard errors are clustered by example because each example appears under several conditions.

Quartiles provide a nonparametric descriptive view of the same relationship. Regression coefficients are converted to odds ratios for interpretation.

In [3]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

association_parts = []
for path in sorted(PATHS['predictions'].glob('**/*.parquet')):
    if '/raw/' not in path.as_posix():
        continue
    frame = pd.read_parquet(path)
    model_name = frame['model'].iloc[0]
    if model_name not in {'bpe_tfidf', 'distilbert'}:
        continue
    tokenizer_name = 'o200k_base' if model_name == 'bpe_tfidf' else 'distilbert_wordpiece'
    clean = frame[frame['attack'] == 'clean'][['example_id', 'correct']].rename(columns={'correct': 'clean_correct'})
    attacked = frame[frame['attack'] != 'clean'].merge(clean, on='example_id', how='left', validate='many_to_one')
    attacked = attacked[attacked['clean_correct'] & (attacked['changed_count'] > 0)].copy()
    metrics = tokenization[tokenization['tokenizer'] == tokenizer_name]
    attacked = attacked.merge(metrics, on=['dataset', 'example_id', 'attack', 'strength', 'attack_seed', 'changed_count'], how='inner', validate='one_to_one')
    attacked['failure'] = (~attacked['correct']).astype(int)
    attacked['tokenizer'] = tokenizer_name
    association_parts.append(attacked[['dataset', 'example_id', 'model', 'model_seed', 'attack', 'strength', 'attack_seed', 'failure', 'normalized_token_edit', 'relative_token_count_change', 'token_multiset_jaccard', 'tokenizer']])

association = pd.concat(association_parts, ignore_index=True)
association['cluster'] = association['dataset'] + ':' + association['example_id']
regression_rows = []
quartile_rows = []
for model_name, group in association.groupby('model'):
    group = group.copy()
    group['z_edit'] = (group['normalized_token_edit'] - group['normalized_token_edit'].mean()) / group['normalized_token_edit'].std(ddof=0)
    group['z_count_change'] = (group['relative_token_count_change'] - group['relative_token_count_change'].mean()) / group['relative_token_count_change'].std(ddof=0)
    seed_term = ' + C(model_seed)' if group['model_seed'].nunique() > 1 else ''
    formula = 'failure ~ z_edit + z_count_change + C(dataset) + C(attack) + C(strength)' + seed_term
    fitted = smf.glm(formula=formula, data=group, family=sm.families.Binomial()).fit(cov_type='cluster', cov_kwds={'groups': group['cluster']})
    confidence = fitted.conf_int()
    for term in ['z_edit', 'z_count_change']:
        regression_rows.append({
            'model': model_name, 'term': term, 'coefficient': fitted.params[term], 'standard_error': fitted.bse[term],
            'p_value': fitted.pvalues[term], 'odds_ratio': math.exp(fitted.params[term]),
            'odds_ratio_ci_low': math.exp(confidence.loc[term, 0]), 'odds_ratio_ci_high': math.exp(confidence.loc[term, 1]),
            'n': int(fitted.nobs), 'formula': formula,
        })
    group['edit_quartile'] = pd.qcut(group['normalized_token_edit'].rank(method='first'), 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
    rates = group.groupby(['dataset', 'edit_quartile'], observed=True)['failure'].agg(['mean', 'size']).reset_index()
    rates['model'] = model_name
    quartile_rows.append(rates.rename(columns={'mean': 'failure_rate', 'size': 'n'}))

regression = pd.DataFrame(regression_rows)
regression.to_csv(PATHS['tables'] / 'token_failure_regression.csv', index=False)
quartiles = pd.concat(quartile_rows, ignore_index=True)
quartiles.to_csv(PATHS['tables'] / 'token_failure_quartiles.csv', index=False)
regression


,model,term,coefficient,standard_error,p_value,odds_ratio,odds_ratio_ci_low,odds_ratio_ci_high,n,formula
0,bpe_tfidf,z_edit,0.969353,0.034568,5.032235e-173,2.636238,2.463543,2.821039,151776,failure ~ z_edit + z_count_change + C(dataset)...
1,bpe_tfidf,z_count_change,0.224145,0.026115,9.233875e-18,1.251253,1.188820,1.316964,151776,failure ~ z_edit + z_count_change + C(dataset)...
2,distilbert,z_edit,1.282606,0.033489,0.000000e+00,3.606024,3.376933,3.850657,438528,failure ~ z_edit + z_count_change + C(dataset)...
3,distilbert,z_count_change,-0.061989,0.018158,6.404961e-04,0.939894,0.907032,0.973946,438528,failure ~ z_edit + z_count_change + C(dataset)...


## Interpretation

An odds ratio above 1 means higher disruption is associated with greater failure odds after the stated controls. The analysis is observational: correlated attack properties may influence both token changes and predictions, so the coefficient is not a causal effect of tokenization alone.